# **Step 1.** Install Python Packages (Unsloth + Colab T4)
Installs Unsloth and compatible PyTorch / xformers dependencies optimized for the free Google Colab T4 GPU.

In [ ]:
# Install Unsloth & required fine-tuning dependencies for Google Colab T4 GPU
# Note: xformers is not manually installed here to avoid setup.py compilation errors on Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-d64_iijg/unsloth_5a2a02231ab54d878e2aad5b59ea0bdf
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-d64_iijg/unsloth_5a2a02231ab54d878e2aad5b59ea0bdf
  Resolved https://github.com/unslothai/unsloth.git to commit 811c08e782cf36af021f558a112e80778e29ba42
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.3 MB/s eta 0:00:00
   ━━

# **Step 2.** Import Python Packages

In [ ]:
from unsloth import FastLanguageModel
import os
import json
import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import notebook_login
from transformers import TrainingArguments
from trl import SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Step 3.** (Optional) Login to Hugging Face
Run this cell if you plan to push your fine-tuned model or dataset to your Hugging Face account.

In [ ]:
# Authenticate with your Hugging Face account (requires a WRITE token)
notebook_login()


# **Step 4.** Load or Generate Diverse ADHIKAAR Dataset
Checks if `train.jsonl` is uploaded. If not, automatically generates 1,000 diverse synthetic samples across 15 Indian states directly in Colab!

In [ ]:
# Check if train.jsonl is uploaded; if not, generate 1000 diverse synthetic samples on the fly
train_path = "train.jsonl"
val_path = "val.jsonl"

if not os.path.exists(train_path):
    for alt in ["ml_training/data/train.jsonl", "/content/train.jsonl"]:
        if os.path.exists(alt):
            train_path = alt
            val_path = alt.replace("train.jsonl", "val.jsonl")
            break

if not os.path.exists(train_path):
    print("train.jsonl not found. Generating diverse ADHIKAAR dataset (1000 train / 200 val)...")
    import random
    FIRST_NAMES = ["Ramesh", "Suresh", "Murugan", "Kavitha", "Priya", "Ananya", "Ramu", "Venkatesh", "Lakshmi", "Rajesh", "Manish", "Sunita", "Deepak", "Pooja", "Selvakumar", "Subba Rao", "Gurpreet", "Sita", "Balaji", "Bhavani"]
    LAST_NAMES = ["Kumar", "Patel", "Singh", "Sharma", "Devi", "Reddy", "Rao", "Gowda", "Nadar", "Pillai", "Yadav", "Verma", "Gupta", "Kaur", "Joshi", "Patil", "Naidu", "Iyer", "Menon", "Bhatt"]
    STATES = [("Tamil Nadu", ["Thanjavur", "Madurai", "Salem"]), ("Andhra Pradesh", ["Guntur", "Krishna", "Kurnool"]), ("Telangana", ["Warangal", "Nalgonda"]), ("Uttar Pradesh", ["Varanasi", "Meerut"]), ("Karnataka", ["Mandya", "Hassan"]), ("Punjab", ["Ludhiana", "Patiala"])]
    BANKS = [("State Bank of India", "SBIN0"), ("Punjab National Bank", "PUNB0"), ("Canara Bank", "CNRB0"), ("Bank of Baroda", "BARB0"), ("Union Bank of India", "UBIN0"), ("HDFC Bank", "HDFC0")]
    SYS_PROMPT = "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. Your job is to extract ONLY explicitly stated facts from the provided text into the requested JSON schema. If a requested field is not mentioned or cannot be verified from the text, you must set its value to null. Output strictly valid JSON with no markdown fences, no preamble, and no explanation."

    def gen_samples(count):
        rows = []
        for _ in range(count):
            kind = random.choice(["aadhaar", "land", "bank", "income", "marks", "decl"])
            name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
            state, dists = random.choice(STATES)
            dist = random.choice(dists)
            if kind == "aadhaar":
                l4 = f"{random.randint(1000, 9999)}"
                txt = f"GOVT OF INDIA - UIDAI\nName: {name}\nDOB: 12/04/1985\nXXXX XXXX {l4}\nAddress: Dist {dist}, {state}"
                prompt = f"Source Type: aadhaar\nTarget Keys: [\"applicant_name\", \"aadhaar_last4\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"applicant_name": name, "aadhaar_last4": l4}
            elif kind == "land":
                ha = round(random.uniform(0.3, 3.5), 2)
                ht = "individual" if random.random() > 0.15 else "institutional"
                txt = f"REVENUE DEPT - GOVT OF {state.upper()}\nROR-1B PATTA\nKhata: {random.randint(100, 999)}\nPattadar: {name if ht=='individual' else 'Gram Panchayat'}\nCategory: {ht.title()}\nExtent: {ha} Hectares"
                prompt = f"Source Type: land_record\nTarget Keys: [\"land_owned_hectares\", \"holder_type\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"land_owned_hectares": ha, "holder_type": ht}
            elif kind == "bank":
                bname, iprefix = random.choice(BANKS)
                ifsc = f"{iprefix}{random.randint(100000, 999999)}"
                acc = f"{random.randint(10000000000, 999999999999)}"
                txt = f"{bname}\nBranch: {dist}\nIFSC: {ifsc}\nAccount Name: {name}\nAccount No: {acc}"
                prompt = f"Source Type: bank_passbook\nTarget Keys: [\"bank_account_number\", \"bank_ifsc\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"bank_account_number": acc, "bank_ifsc": ifsc}
            elif kind == "income":
                inc = random.choice([48000, 60000, 85000, 120000, 180000, 240000])
                txt = f"TALUK OFFICE {dist.upper()}\nINCOME CERTIFICATE\nAnnual income of {name} is Rs. {inc:,}/- per annum."
                prompt = f"Source Type: income_certificate\nTarget Keys: [\"family_income_annual\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"family_income_annual": float(inc)}
            elif kind == "marks":
                pct = round(random.uniform(55.0, 95.0), 1)
                txt = f"BOARD OF SECONDARY EDUCATION {state.upper()}\nCandidate: {name}\nAggregate Percentage: {pct}%"
                prompt = f"Source Type: marksheet\nTarget Keys: [\"last_exam_percentage\"]\n\nDocument OCR Text:\n{txt}"
                exp = {"last_exam_percentage": pct}
            else:
                if random.random() > 0.4:
                    txt = "Namaste, I am a small cultivator. Nobody in my family pays any income tax."
                    exp = {"is_income_tax_payer": False}
                else:
                    txt = "Hello sir, here are my land records and Aadhaar for the farmer scheme."
                    exp = {"is_income_tax_payer": None}
                prompt = f"Source Type: declaration\nTarget Keys: [\"is_income_tax_payer\"]\n\nCitizen Voice Transcript:\n{txt}"
            rows.append({"messages": [
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": json.dumps(exp)}
            ]})
        return rows

    with open("train.jsonl", "w", encoding="utf-8") as f:
        for item in gen_samples(1000):
            f.write(json.dumps(item) + "\n")
    with open("val.jsonl", "w", encoding="utf-8") as f:
        for item in gen_samples(200):
            f.write(json.dumps(item) + "\n")
    train_path = "train.jsonl"
    val_path = "val.jsonl"

dataset = load_dataset("json", data_files={"train": train_path, "validation": val_path})
print(f"Loaded dataset: {len(dataset['train'])} train rows, {len(dataset['validation'])} val rows.")


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loaded dataset: 1000 train rows, 200 val rows.


# **Step 5.** LoRA & Hardware Settings (Optimized for Colab T4 GPU)
- Base model: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
- Uses 4-bit quantization (~5.5GB VRAM footprint)
- `fp16=True`, `bf16=False` (T4 supports FP16, not BF16)
- LoRA rank 16 on all linear attention + MLP projection layers

In [ ]:
# Hardware & LoRA Hyperparameters for Meta-Llama-3.1-8B-Instruct on T4
config = {
    "huggingface_user": "YOUR_HF_USERNAME",  # Replace with your Hugging Face username
    "model_config": {
        "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", # Llama-3.1-8B 4-bit
        "finetuned_model": "adhikaar-llama3.1-8b-fact-extractor",     # Model output name
        "max_seq_length": 2048,                                      # Max tokens per sample
        "dtype": None,                                               # Auto-detects float16 on T4
        "load_in_4bit": True,                                        # 4-bit QLoRA
    },
    "lora_config": {
        "r": 16,                                                     # LoRA rank
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "lora_alpha": 16,
        "lora_dropout": 0,                                           # 0 is optimized by Unsloth
        "bias": "none",
        "use_gradient_checkpointing": "unsloth",                     # Saves 30% VRAM
        "random_state": 42,
    },
    "training_config": {
        "per_device_train_batch_size": 2,                            # Fits easily in 16GB VRAM
        "gradient_accumulation_steps": 4,                            # Effective batch size = 8
        "warmup_steps": 10,
        "num_train_epochs": 3,                                       # 3 epochs avoids overfitting
        "learning_rate": 2e-4,
        "fp16": True,                                                # T4 supports fp16
        "bf16": False,                                               # T4 does NOT support bf16
        "logging_steps": 10,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 42,
        "output_dir": "outputs",
    }
}
print("Configuration ready for Llama-3.1-8B on T4.")


Configuration ready for Llama-3.1-8B on T4.


# **Step 6.** Load Llama-3.1-8B, QLoRA & SFTTrainer

In [ ]:
# 1. Load Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["model_config"]["base_model"],
    max_seq_length=config["model_config"]["max_seq_length"],
    dtype=config["model_config"]["dtype"],
    load_in_4bit=config["model_config"]["load_in_4bit"],
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora_config"]["r"],
    target_modules=config["lora_config"]["target_modules"],
    lora_alpha=config["lora_config"]["lora_alpha"],
    lora_dropout=config["lora_config"]["lora_dropout"],
    bias=config["lora_config"]["bias"],
    use_gradient_checkpointing=config["lora_config"]["use_gradient_checkpointing"],
    random_state=config["lora_config"]["random_state"],
)

# 3. Format dataset using Llama-3.1 native chat template
def format_chatml_prompts(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_data = dataset["train"].map(format_chatml_prompts, batched=True)
val_data = dataset["validation"].map(format_chatml_prompts, batched=True)

# 4. Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=config["model_config"]["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=config["training_config"]["per_device_train_batch_size"],
        gradient_accumulation_steps=config["training_config"]["gradient_accumulation_steps"],
        warmup_steps=config["training_config"]["warmup_steps"],
        num_train_epochs=config["training_config"]["num_train_epochs"],
        learning_rate=config["training_config"]["learning_rate"],
        fp16=config["training_config"]["fp16"],
        bf16=config["training_config"]["bf16"],
        logging_steps=config["training_config"]["logging_steps"],
        optim=config["training_config"]["optim"],
        weight_decay=config["training_config"]["weight_decay"],
        lr_scheduler_type=config["training_config"]["lr_scheduler_type"],
        seed=config["training_config"]["seed"],
        output_dir=config["training_config"]["output_dir"],
        report_to="none",
    ),
)
print("SFTTrainer successfully initialized.")


==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

SFTTrainer successfully initialized.


# **Step 7.** Train Your Fine-Tuned Model
Runs fine-tuning for 3 epochs (~15–20 minutes on Colab T4).

In [ ]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 3 | Total steps = 375
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.087096
20,1.029047
30,0.501561
40,0.303624
50,0.285751
60,0.243943
70,0.247881
80,0.221448
90,0.226007
100,0.226249


Step,Training Loss
10,3.087096
20,1.029047
30,0.501561
40,0.303624
50,0.285751
60,0.243943
70,0.247881
80,0.221448
90,0.226007
100,0.226249


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-375/tokenizer_config.json.


# **Step 8.** Save Trainer Stats & VRAM Metrics

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 2)
total_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 2)
runtime_sec = trainer_stats.metrics.get("train_runtime", 0)

print(f"Training Runtime: {runtime_sec / 60:.2f} minutes")
print(f"Peak VRAM Used: {max_memory} GB / {total_memory} GB ({round(max_memory / total_memory * 100, 1)}%)")

with open("trainer_stats.json", "w") as f:
    json.dump(trainer_stats.metrics, f, indent=4)


Training Runtime: 44.06 minutes
Peak VRAM Used: 6.2 GB / 14.56 GB (42.6%)


# **Step 9.** Save Model & Export Options
- Option A: Save LoRA adapters locally
- Option B: Push LoRA adapters to Hugging Face
- Option C: Push merged 16-bit model for Hugging Face Inference Endpoints / vLLM
- Option D: Push GGUF for local Ollama

In [ ]:
model_name = config["model_config"]["finetuned_model"]
hf_user = config["huggingface_user"]

# 1. Save LoRA adapter locally
model.save_pretrained(f"{model_name}-lora")
tokenizer.save_pretrained(f"{model_name}-lora")
print(f"LoRA adapter saved to {model_name}-lora")

# 2. Push LoRA adapter to Hugging Face Hub (Uncomment if logged in)
# model.push_to_hub(f"{hf_user}/{model_name}", tokenizer=tokenizer)

# 3. Push merged 16-bit model (Uncomment for Hugging Face Inference Endpoints / vLLM)
# model.push_to_hub_merged(f"{hf_user}/{model_name}-merged-16bit", tokenizer=tokenizer, save_method="merged_16bit")

# 4. Push GGUF quantized model (Uncomment for Ollama / llama.cpp)
# model.push_to_hub_gguf(f"{hf_user}/{model_name}-GGUF", tokenizer=tokenizer, quantization_method="q4_k_m")


Unsloth: Restored added_tokens_decoder metadata in adhikaar-llama3.1-8b-fact-extractor-lora/tokenizer_config.json.


LoRA adapter saved to adhikaar-llama3.1-8b-fact-extractor-lora


# **Step 10.** Test Fact Extraction Inference on ADHIKAAR Documents
Validates that the fine-tuned model extracts structured facts into strictly valid JSON.

In [ ]:
import torch
from transformers import TextStreamer
from unsloth import FastLanguageModel

# 1. Clean residual training tensors from VRAM
torch.cuda.empty_cache()

# 2. Enable Unsloth native 2x fast inference kernels
FastLanguageModel.for_inference(model)

# 3. Setup streaming so you see tokens appear in real-time
streamer = TextStreamer(tokenizer, skip_prompt=True)

# 4. Explicitly tell Llama-3.1 to STOP on <|eot_id|> (ID: 128009)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

test_prompt = """Source Type: land_record
Target Keys: ["land_owned_hectares", "holder_type"]

Document OCR Text:
REVENUE DEPARTMENT - GOVT OF TAMIL NADU
EXTRACT OF REGISTER OF RIGHTS (ROR-1B / PATTA)
Pattadar Name: Murugan K
Holder Category: Individual Cultivator
Total Extent: 1.40 Hectares"""

system_prompt = (
    "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. "
    "Your job is to extract ONLY explicitly stated facts into the requested JSON schema. "
    "Output strictly valid JSON with no markdown fences, no preamble, and no explanation."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("--- Fast Streaming Response ---")
with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        streamer=streamer,              # Streams tokens instantly
        max_new_tokens=64,              # JSON only needs ~30-40 tokens max
        do_sample=False,                # Greedy search is fastest & deterministic
        eos_token_id=terminators,       # STOPS immediately when JSON completes
        use_cache=True,
    )


--- Fast Streaming Response ---


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{"land_owned_hectares": 1.4, "holder_type": "individual"}<|eot_id|>


In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copytree("adhikaar-llama3.1-8b-fact-extractor-lora", "/content/drive/MyDrive/adhikaar-llama3.1-8b-fact-extractor-lora")
print("Safely backed up to Google Drive!")


Mounted at /content/drive
Safely backed up to Google Drive!


In [ ]:
# 1. Save LoRA adapter locally (THIS RAN — saved inside Colab)
model.save_pretrained(f"{model_name}-lora")

# 2. Push LoRA adapter to Hugging Face Hub (THIS WAS COMMENTED OUT)
# model.push_to_hub(f"{hf_user}/{model_name}", tokenizer=tokenizer)


In [ ]:
from huggingface_hub import login

# 1. Login with a WRITE token (generate one at https://huggingface.co/settings/tokens)
login()

# 2. Set your Hugging Face username
HF_USERNAME = "dheeraj013"  # <-- Replace with your real HF username
MODEL_NAME = "adhikaar-llama3.1-8b-fact-extractor"

# 3. Upload LoRA adapter and tokenizer separately
model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

print(f"Done! View your model here: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

Validating                    :           |   0%            

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp3p_iq7hq/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Done! View your model here: https://huggingface.co/dheeraj013/adhikaar-llama3.1-8b-fact-extractor


In [ ]:
# Define the new separate name with -lora suffix
SEPARATE_MODEL_NAME = "adhikaar-llama3.1-8b-fact-extractor-lora"

# Push the LoRA adapters to this new repository name
print(f"Uploading LoRA adapter to Hugging Face as a separate model: {HF_USERNAME}/{SEPARATE_MODEL_NAME}...")
model.push_to_hub(f"{HF_USERNAME}/{SEPARATE_MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{SEPARATE_MODEL_NAME}")

print(f"Successfully uploaded! View your separate LoRA adapter model here: https://huggingface.co/{HF_USERNAME}/{SEPARATE_MODEL_NAME}")

Uploading LoRA adapter to Hugging Face as a separate model: dheeraj013/adhikaar-llama3.1-8b-fact-extractor-lora...


README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|9         | 15.9MB /  168MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpf3lgx30y/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

Validating                    :           |   0%            

Successfully uploaded! View your separate LoRA adapter model here: https://huggingface.co/dheeraj013/adhikaar-llama3.1-8b-fact-extractor-lora


In [ ]:
import torch
from transformers import TextStreamer
from unsloth import FastLanguageModel

# 1. Clean residual training tensors from VRAM
torch.cuda.empty_cache()

# 2. Enable Unsloth native 2x fast inference kernels
FastLanguageModel.for_inference(model)

# 3. Setup streaming so you see tokens appear in real-time
streamer = TextStreamer(tokenizer, skip_prompt=True)

# 4. Explicitly tell Llama-3.1 to STOP on <|eot_id|> (ID: 128009)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

test_prompt = """Source Type: land_record
Target Keys: ["land_owned_hectares", "holder_type"]

Document OCR Text:
REVENUE DEPARTMENT - GOVT OF TAMIL NADU
EXTRACT OF REGISTER OF RIGHTS (ROR-1B / PATTA)
Pattadar Name: Murugan K
Holder Category: Individual Cultivator
Total Extent: 1.40 Hectares"""

system_prompt = (
    "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. "
    "Your job is to extract ONLY explicitly stated facts into the requested JSON schema. "
    "Output strictly valid JSON with no markdown fences, no preamble, and no explanation."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("--- Fast Streaming Response ---")
with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        streamer=streamer,              # Streams tokens instantly
        max_new_tokens=64,              # JSON only needs ~30-40 tokens max
        do_sample=False,                # Greedy search is fastest & deterministic
        eos_token_id=terminators,       # STOPS immediately when JSON completes
        use_cache=True,
    )


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Fast Streaming Response ---
{"land_owned_hectares": 1.4, "holder_type": "individual"}<|eot_id|>


### **Test the Fine-Tuned Model from Hugging Face**
This cell loads the fine-tuned LoRA adapters directly from Hugging Face (`dheeraj013/adhikaar-llama3.1-8b-fact-extractor-lora`) and runs inference on a test sample to verify its performance.

In [ ]:
import torch
from transformers import TextStreamer
from unsloth import FastLanguageModel

# 1. Clear memory
torch.cuda.empty_cache()

# 2. Load the base model and your separate LoRA adapters from Hugging Face
HuggingFace_model_id = "dheeraj013/adhikaar-llama3.1-8b-fact-extractor-lora"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HuggingFace_model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# 3. Enable 2x faster inference
FastLanguageModel.for_inference(model)

# 4. Create a fresh test document
test_doc = """Source Type: income_certificate
Target Keys: ["family_income_annual"]

Document OCR Text:
OFFICE OF THE TAHSILDAR - GUNTUR
INCOME CERTIFICATE
This is to certify that the annual income from all sources of Rajesh Reddy is Rs. 75,000/- (Seventy Five Thousand Rupees only)."""

# 5. Build prompt using ChatML template
messages = [
    {"role": "system", "content": "You are ADHIKAAR's structured fact extraction engine for Indian welfare schemes. Your job is to extract ONLY explicitly stated facts into the requested JSON schema. Output strictly valid JSON with no markdown fences, no preamble, and no explanation."},
    {"role": "user", "content": test_doc},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 6. Stream the model's response
streamer = TextStreamer(tokenizer, skip_prompt=True)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

print("\n--- Model Output (Streaming) ---")
with torch.inference_mode():
    _ = model.generate(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=64,
        do_sample=False,
        eos_token_id=terminators,
        use_cache=True,
    )

==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Model Output (Streaming) ---
{"family_income_annual": 75000.0}<|eot_id|>


### **Export & Run Fine-Tuned Model in LM Studio**

LM Studio requires models to be in the **GGUF** format. Since you have fine-tuned a LoRA adapter, you must merge the adapters back into the 16-bit or 4-bit base model and export it as GGUF.

Run the cell below to merge your model and save it locally as a 4-bit GGUF file (highly recommended for fast local inference).

In [ ]:
# 1. Install Unsloth and dependencies if they are missing in this session
try:
    from unsloth import FastLanguageModel
except ModuleNotFoundError:
    print("Unsloth not found. Installing dependencies (this takes about 1 minute)...")
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
    !pip install --no-deps trl peft accelerate bitsandbytes --quiet
    from unsloth import FastLanguageModel

# 2. Load your fine-tuned model and tokenizer from Hugging Face
HuggingFace_model_id = "dheeraj013/adhikaar-llama3.1-8b-fact-extractor-lora"

print("Loading fine-tuned model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HuggingFace_model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# 3. Merge LoRA adapters and save as a Q4_K_M GGUF file locally in Colab
print("Starting GGUF quantization (this may take 2-5 minutes)...")
model.save_pretrained_gguf(
    "adhikaar-llama3.1-8b-fact-extractor-GGUF",
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF quantization complete!")

Unsloth not found. Installing dependencies (this takes about 1 minute)...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.4 MB/s eta 0:00:00
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsl

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
Starting GGUF quantization (this may take 2-5 minutes)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:55<00:00, 103.97s/it]


Unsloth: Merge process complete. Saved to `/content/adhikaar-llama3.1-8b-fact-extractor-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11734-mix-c208e5f (app-b11734-mix-c208e5f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model 

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Modelfile
GGUF quantization complete!


In [ ]:
import os
import shutil

source_path = "/content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf"
destination_dir = "/content/drive/MyDrive/adhikaar_gguf_model"
destination_path = os.path.join(destination_dir, "Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf")

if os.path.exists(source_path):
    print(f"Found GGUF file locally at: {source_path}")
    os.makedirs(destination_dir, exist_ok=True)
    print("Copying GGUF file to Google Drive (this will take a couple of minutes)...")
    shutil.copy(source_path, destination_path)
    print(f"\nSuccess! The file is backed up on your Google Drive at: Drive -> MyDrive -> adhikaar_gguf_model -> Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf")
else:
    print("Please run the GGUF quantization cell (908b920c) completely first to generate the file!")

Found GGUF file locally at: /content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf
Copying GGUF file to Google Drive (this will take a couple of minutes)...

Success! The file is backed up on your Google Drive at: Drive -> MyDrive -> adhikaar_gguf_model -> Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf


### **Upload the GGUF Model to Hugging Face (Make it Searchable in LM Studio)**

To make your model visible globally in LM Studio's built-in search bar, we will create a new repository on Hugging Face specifically for this GGUF file and upload it.

In [ ]:
import getpass
from huggingface_hub import HfApi, create_repo, login

# 1. Securely request your Hugging Face write token
hf_token = getpass.getpass("Paste your Hugging Face WRITE token here (input will be hidden): ")

try:
    # Log in using the provided write token
    login(token=hf_token)

    # Configure your settings
    HF_USERNAME = "dheeraj013"
    GGUF_REPO_NAME = "adhikaar-llama3.1-8b-fact-extractor-GGUF"
    local_file_path = "/content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf"

    api = HfApi()

    # 2. Create a new model repository on Hugging Face for the GGUF file
    repo_id = f"{HF_USERNAME}/{GGUF_REPO_NAME}"
    print(f"\nCreating repository: {repo_id}...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=hf_token)

    # 3. Upload the GGUF file to the new repository
    print(f"Uploading {local_file_path} to Hugging Face (this will take a few minutes)...")
    api.upload_file(
        path_or_fileobj=local_file_path,
        path_in_repo="Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf",
        repo_id=repo_id,
        repo_type="model",
        token=hf_token
    )
    print(f"\nSuccess! Your GGUF model is now uploaded to Hugging Face: https://huggingface.co/{repo_id}")
    print("People can now find your model in LM Studio by searching for 'dheeraj013' or 'adhikaar-llama3.1'!")

except Exception as e:
    print(f"\nAn error occurred during login or upload: {e}")

Paste your Hugging Face WRITE token here (input will be hidden): ··········

Creating repository: dheeraj013/adhikaar-llama3.1-8b-fact-extractor-GGUF...
Uploading /content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf to Hugging Face (this will take a few minutes)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1-8B-Instruct.Q4_K_M.gguf:   0%|          |  559kB / 4.92GB            

Validating                    :           |   0%            


Success! Your GGUF model is now uploaded to Hugging Face: https://huggingface.co/dheeraj013/adhikaar-llama3.1-8b-fact-extractor-GGUF
People can now find your model in LM Studio by searching for 'dheeraj013' or 'adhikaar-llama3.1'!


In [ ]:
import getpass
from huggingface_hub import HfApi, create_repo, login

# 1. Reuse the token or request it again if the session restarted
try:
    token = hf_token
except NameError:
    token = getpass.getpass("Paste your Hugging Face WRITE token: ")

# Log in
login(token=token)

# 2. Configure the new distinctive repository name
HF_USERNAME = "dheeraj013"
DISTINCT_REPO_NAME = "adhikaar-llama3.1-8b-fact-extractor-GGUF-Q4"
local_file_path = "/content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf"

api = HfApi()
repo_id = f"{HF_USERNAME}/{DISTINCT_REPO_NAME}"

print(f"\nCreating a completely distinct repository: {repo_id}...")
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=token)

print(f"Uploading {local_file_path} (this will take a few minutes)...")
api.upload_file(
    path_or_fileobj=local_file_path,
    path_in_repo="Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model",
    token=token
)

print(f"\nSuccess! Your GGUF model is now uploaded to: https://huggingface.co/{repo_id}")


Creating a completely distinct repository: dheeraj013/adhikaar-llama3.1-8b-fact-extractor-GGUF-Q4...
Uploading /content/adhikaar-llama3.1-8b-fact-extractor-GGUF_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf (this will take a few minutes)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1-8B-Instruct.Q4_K_M.gguf:   0%|          | 7.97MB / 4.92GB            

Validating                    :           |   0%            


Success! Your GGUF model is now uploaded to: https://huggingface.co/dheeraj013/adhikaar-llama3.1-8b-fact-extractor-GGUF-Q4


### **How to Download and Load the GGUF in LM Studio**

#### **Step 1: Download the `.gguf` file**
1. In Google Colab, click on the **Files** folder icon in the left-hand sidebar.
2. Navigate to the folder named `adhikaar-llama3.1-8b-fact-extractor-GGUF`.
3. Locate the file ending with `.gguf` (e.g., `unsloth.Q4_K_M.gguf`).
4. Click the three dots next to it and select **Download** (it will be around 4.8 GB, so it may take a few minutes to download to your computer).

#### **Step 2: Move the GGUF file to LM Studio's Local Directory**
LM Studio detects local files placed inside its designated folder structure.

1. Open your local file explorer/finder and go to your **User Profile directory**:
   - **Windows**: `C:\Users\<Your-Username>\.cache\lm-studio\models\`
   - **Mac/Linux**: `~/.cache/lm-studio/models/`
2. Inside the `models` folder, create a new subfolder structure indicating the creator and model name (e.g., `local/adhikaar-llama3.1/`).
3. Paste the downloaded `.gguf` file inside that folder. For example:
   `~/.cache/lm-studio/models/local/adhikaar-llama3.1/unsloth.Q4_K_M.gguf`

#### **Step 3: Load the model in LM Studio**
1. Launch **LM Studio** on your computer.
2. Go to the **Chat** tab (the bubble icon on the left).
3. Click on the drop-down menu at the top center (**"Select a model to load"**).
4. Your newly added model (`unsloth.Q4_K_M.gguf`) will appear under the **Local Models** list. Click to load it!
5. Paste your system prompt and test prompts in the chat box to begin running local structured fact extraction.